# K-Fold Training v2 (No AMP + Dtype Debug)

This notebook intentionally disables AMP/autocast/GradScaler.
It adds explicit dtype/device debug prints and a one-step smoke test before full training.


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

REPO_URL = "https://github.com/mruniverse8/kaggle-experiments-.git"
REPO_DIR = Path("/kaggle/working/kaggle-experiments-")
BRANCH = "twitter_sentiment_v2_noamp_debug"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "--all"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

current_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
current_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print("Git branch:", current_branch)
print("Git commit:", current_commit)
print("Repo ready at:", REPO_DIR)


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

import sys
sys.path.insert(0, str(Path("src").resolve()))

from dz2_causal.dataset import (
    TweetExtractionCausalDataset,
    causal_collate_fn,
    register_special_tokens,
)
from dz2_causal.losses import compute_total_loss
from dz2_causal.modeling import CausalExtractionModel
from dz2_causal.eval_utils import evaluate_dataframe_jaccard

CFG_PATH = os.environ.get("CFG_PATH", "config/kaggle_train_kfold_small_16gb.json")
print("Using config:", CFG_PATH)
CFG = json.loads(Path(CFG_PATH).read_text())
CFG

assert bool(CFG.get("no_amp", False)) is True, "For v2 debug, set no_amp=true in config."
print("no_amp:", CFG["no_amp"], "| model:", CFG["model_name"])


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])

df = pd.read_csv(CFG["train_csv"]).dropna(subset=["text", "selected_text"]).reset_index(drop=True)
print("Train rows:", len(df))

splits = list(
    StratifiedKFold(
        n_splits=CFG["n_splits"],
        shuffle=True,
        random_state=CFG["seed"],
    ).split(df, df["sentiment"])
)

OUTPUT_DIR = Path(CFG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR)


In [ ]:
def make_records(frame: pd.DataFrame):
    records = frame[["text", "sentiment", "selected_text"]].to_dict("records")
    for r in records:
        r["prompt"] = CFG["prompt_text"]
    return records


def print_dtype_report(model, batch, prefix=""):
    emb_dtype = model.lm.get_input_embeddings().weight.dtype
    print(f"{prefix} input_ids dtype={batch['input_ids'].dtype} device={batch['input_ids'].device}")
    print(f"{prefix} attention_mask dtype={batch['attention_mask'].dtype} device={batch['attention_mask'].device}")
    print(f"{prefix} labels dtype={batch['labels'].dtype} device={batch['labels'].device}")
    print(f"{prefix} source_mask dtype={batch['source_mask'].dtype} device={batch['source_mask'].device}")
    print(f"{prefix} LM embedding dtype={emb_dtype}")
    print(f"{prefix} start_head weight dtype={model.start_head.weight.dtype}")
    print(f"{prefix} end_head weight dtype={model.end_head.weight.dtype}")
    print(f"{prefix} select_head weight dtype={model.select_head.weight.dtype}")


def align_head_dtypes_with_lm(model, device):
    target_dtype = model.lm.get_input_embeddings().weight.dtype
    model.start_head.to(device=device, dtype=target_dtype)
    model.end_head.to(device=device, dtype=target_dtype)
    model.select_head.to(device=device, dtype=target_dtype)
    return target_dtype


def run_smoke_step(model, loader, optimizer, scheduler, device):
    smoke_batch = next(iter(loader))
    for k, v in smoke_batch.items():
        if torch.is_tensor(v):
            smoke_batch[k] = v.to(device)

    if CFG.get("debug_dtype_print", True):
        print_dtype_report(model, smoke_batch, prefix="[smoke]")

    out = model(
        input_ids=smoke_batch["input_ids"],
        attention_mask=smoke_batch["attention_mask"],
        labels=smoke_batch["labels"],
        compute_span_logits=False,
    )
    losses = compute_total_loss(
        outputs=out,
        batch=smoke_batch,
        lambda_kl=0.0,
        lambda_select=0.0,
    )
    loss = losses["loss"]
    print(f"[smoke] loss={loss.item():.4f} ce={losses['ce_loss'].item():.4f}")
    print(f"[smoke] start_logits dtype={out['start_logits'].dtype}")
    print(f"[smoke] end_logits dtype={out['end_logits'].dtype}")
    print(f"[smoke] select_logits dtype={out['select_logits'].dtype}")

    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()


def train_one_fold(fold_id: int, train_idx, val_idx):
    print(f"\n===== Fold {fold_id} =====")

    tokenizer = AutoTokenizer.from_pretrained(
        CFG["model_name"],
        use_fast=True,
        trust_remote_code=CFG.get("trust_remote_code", False),
    )

    model = CausalExtractionModel(
        model_name=CFG["model_name"],
        trust_remote_code=CFG.get("trust_remote_code", False),
    )
    register_special_tokens(tokenizer, model=model.lm)

    train_ds = TweetExtractionCausalDataset(
        records=make_records(df.iloc[train_idx].reset_index(drop=True)),
        tokenizer=tokenizer,
        prompt_text=CFG["prompt_text"],
        max_len=CFG["max_len"],
        soft_alpha=CFG["soft_alpha"],
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=CFG["batch_size"],
        shuffle=True,
        num_workers=CFG["num_workers"],
        collate_fn=lambda b: causal_collate_fn(b, tokenizer.pad_token_id),
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    lm_dtype = align_head_dtypes_with_lm(model, device)
    print(f"Aligned heads to LM dtype: {lm_dtype}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
    )

    total_steps = max(1, CFG["epochs"] * len(train_loader))
    warmup_steps = int(CFG["warmup_ratio"] * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    optimizer.zero_grad(set_to_none=True)

    if CFG.get("run_smoke_test", True):
        run_smoke_step(model, train_loader, optimizer, scheduler, device)

    model.train()
    for epoch in range(CFG["epochs"]):
        ce_only = epoch < CFG["ce_only_epochs"]
        lambda_kl = 0.0 if ce_only else CFG["lambda_kl"]
        lambda_select = 0.0 if ce_only else CFG["lambda_select"]
        need_span = (lambda_kl > 0.0) or (lambda_select > 0.0)

        running = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "sel": 0.0}
        for step, batch in enumerate(train_loader):
            for k, v in batch.items():
                if torch.is_tensor(v):
                    batch[k] = v.to(device)

            if (
                CFG.get("debug_dtype_print", False)
                and step % int(CFG.get("debug_print_every_steps", 100)) == 0
                and epoch == 0
            ):
                print_dtype_report(model, batch, prefix=f"[fold {fold_id} epoch {epoch+1} step {step+1}]")

            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
                compute_span_logits=need_span,
            )
            losses = compute_total_loss(
                out,
                batch,
                lambda_kl=lambda_kl,
                lambda_select=lambda_select,
            )
            loss = losses["loss"] / CFG["grad_accum_steps"]
            loss.backward()

            if (step + 1) % CFG["grad_accum_steps"] == 0:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            running["loss"] += losses["loss"].item()
            running["ce"] += losses["ce_loss"].item()
            running["kl"] += losses["kl_loss"].item()
            running["sel"] += losses["select_loss"].item()

            if (step + 1) % CFG["log_every"] == 0:
                denom = float(CFG["log_every"])
                print(
                    f"fold={fold_id} epoch={epoch+1} step={step+1}/{len(train_loader)} "
                    f"loss={running['loss']/denom:.4f} ce={running['ce']/denom:.4f} "
                    f"kl={running['kl']/denom:.4f} sel={running['sel']/denom:.4f}"
                )
                running = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "sel": 0.0}

    val_df = df.iloc[val_idx].reset_index(drop=True)
    eval_out = evaluate_dataframe_jaccard(
        df=val_df,
        model=model,
        tokenizer=tokenizer,
        prompt_text=CFG["prompt_text"],
        device=device,
        max_new_tokens=CFG["max_new_tokens"],
    )

    ckpt_path = OUTPUT_DIR / f"model_fold{fold_id}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_name": CFG["model_name"],
            "fold": fold_id,
            "val_jaccard": eval_out["mean_jaccard"],
            "config": CFG,
        },
        ckpt_path,
    )

    print(f"Fold {fold_id} val_jaccard={eval_out['mean_jaccard']:.4f} | saved: {ckpt_path}")
    return {
        "fold": fold_id,
        "val_jaccard": eval_out["mean_jaccard"],
        "checkpoint": str(ckpt_path),
    }


In [ ]:
fold_results = []
for fold_id, (train_idx, val_idx) in enumerate(splits):
    result = train_one_fold(fold_id, train_idx, val_idx)
    fold_results.append(result)

    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_path = OUTPUT_DIR / "kfold_metrics.json"
metrics_path.write_text(json.dumps(fold_results, indent=2))
print("Saved metrics:", metrics_path)
fold_results


In [ ]:
scores = [r["val_jaccard"] for r in fold_results]
print("Mean Jaccard:", float(np.mean(scores)))
print("Std Jaccard:", float(np.std(scores)))
